# Modelling Table
This notebook combines the cleaned rental data, property-level spatial features and SA2-level external features into a single modelling-ready dataset.

## Load Data

In [ ]:
import pandas as pd

domain_df = pd.read_parquet("../data/analysis_data/")
sa2_df = pd.read_parquet("../data/curated/sa2_features.parquet")
spatial_df = pd.read_parquet("../data/curated/property_geo_features.parquet")

In [ ]:
print(domain_df.columns.tolist())
print(spatial_df.columns.tolist())
print(sa2_df.columns.tolist())

## Validate Join Keys

In [ ]:
print(domain_df["listing_id"].dtype)
print(spatial_df["property_id"].dtype)

print(spatial_df["sa2_code_2021"].dtype)
print(sa2_df["sa2_code_2021"].dtype)

In [ ]:
print("Domain listing_id duplicates:",
        domain_df["listing_id"].duplicated().sum())

print("Spatial property_id duplicates:",
        spatial_df["property_id"].duplicated().sum())

print("SA2 code duplicates:",
        sa2_df["sa2_code_2021"].duplicated().sum())

## Merge Data

In [ ]:
# Align join keys
domain_df["listing_id"] = domain_df["listing_id"].astype(str)
spatial_df["property_id"] = spatial_df["property_id"].astype(str)

spatial_df["sa2_code_2021"] = spatial_df["sa2_code_2021"].astype(str)
sa2_df["sa2_code_2021"] = sa2_df["sa2_code_2021"].astype(str)

# Domain + property-level spatial features
master_df = domain_df.merge(
    spatial_df,
    left_on="listing_id",
    right_on="property_id",
    how="left",
    validate="one_to_one",
    indicator="spatial_merge_status"
)

print("After spatial merge:", master_df.shape)
print(master_df["spatial_merge_status"].value_counts())

assert (master_df["spatial_merge_status"] == "both").all()

In [ ]:
master_df = master_df.merge(
    sa2_df,
    on="sa2_code_2021",
    how="left",
    validate="many_to_one",
    suffixes=("", "_sa2"),
    indicator="sa2_merge_status"
)

print("Final master shape:", master_df.shape)
print(master_df["sa2_merge_status"].value_counts())

assert (master_df["sa2_merge_status"] == "both").all()

## Quality Checks

In [ ]:
print("Rows:", len(master_df))
print("Unique listings:", master_df["listing_id"].nunique())

print("Missing SA2 match:",
        master_df["sa2_code_2021"].isna().sum())

print("Missing population_2026 values:",
        master_df["population_2026"].isna().sum())

In [ ]:
print("Master shape:", master_df.shape)

missing = (master_df.isna().sum().sort_values(ascending=False))

print(missing[missing > 0])

In [ ]:
# Check ranges of key modelling variables
cols = [
    "weekly_rent",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "route_km_to_cbd",
    "route_min_to_cbd",
    "straight_km_to_nearest_station",
    "income_median",
    "population_growth_5y"
]

print(master_df[cols].describe())

In [ ]:
print(master_df["is_modelling_ready"].value_counts(dropna=False))
print(master_df["is_analysable"].value_counts(dropna=False))

In [ ]:
excluded_df = master_df[~master_df["is_modelling_ready"]][["listing_id", "suburb", "sa2_code_2021", "sa2_name_2021", "is_analysable", "is_modelling_ready", "is_residential", "income_reliable"]]

excluded_df

In [ ]:
assert master_df["listing_id"].is_unique
assert master_df["sa2_code_2021"].notna().all()
assert master_df["population_2026"].notna().all()
assert (master_df["listing_id"] == master_df["property_id"]).all()
assert (master_df["sa2_name_2021"] == master_df["sa2_name_2021_sa2"]).all()

In [ ]:
master_df = master_df.drop(columns=["property_id", "sa2_name_2021_sa2", "spatial_merge_status", "sa2_merge_status"])

## Output

In [ ]:
# Full merged table
master_df.to_parquet("../data/curated/master_modelling_table_full.parquet",
                        index=False)

# Modelling-ready sample only
modelling_df = master_df[master_df["is_modelling_ready"]].copy()

modelling_df.to_parquet("../data/curated/master_modelling_table.parquet",
                        index=False)

modelling_df.to_csv("../data/curated/master_modelling_table.csv", index=False)

In [ ]:
print("Full:", master_df.shape)
print("Modelling-ready:", modelling_df.shape)